# Forward and backward chaining

This notebook goes with the Detailed lesson **Knowledge Representation and Expert Systems**, section 5.

We build the Mars rover's rule base, then reason with it in both directions. Run each cell in order with **Shift + Enter**.

## 1. The knowledge base

Each rule is (conditions, conclusion). A rule fires when all its conditions are known facts.

In [ ]:
RULES = [
    ({"panels dusty", "daytime"}, "power degraded"),                      # rule 1
    ({"power degraded", "battery low"}, "safe mode"),                     # rule 2
    ({"dish misaligned"}, "communication lost"),                          # rule 3
    ({"safe mode", "communication lost"}, "emergency halt"),              # rule 4
    ({"battery low", "night"}, "hibernate"),                              # rule 5
    ({"battery low", "daytime", "panels clean"}, "recharge"),             # rule 6
]

## 2. Forward chaining

Start from the sensor facts. On each pass, fire every rule whose conditions hold and add its conclusion. Stop when a pass adds nothing new.

In [ ]:
def forward_chain(facts):
    facts = set(facts)
    step = 1
    while True:
        new = []
        for number, (conditions, conclusion) in enumerate(RULES, start=1):
            if conditions <= facts and conclusion not in facts:
                new.append((number, conclusion))
        if not new:
            print(f"Pass {step}: nothing new, so stop.")
            return facts
        for number, conclusion in new:
            print(f"Pass {step}: rule {number} fires, giving '{conclusion}'")
        facts |= {c for _, c in new}
        step += 1

working_memory = forward_chain({"panels dusty", "daytime", "battery low", "dish misaligned"})
print()
print("Working memory now holds:", sorted(working_memory))

## 3. Backward chaining

Start from a goal. If it is a known fact, done. Otherwise find a rule that concludes it and prove that rule's conditions as sub-goals. If no rule concludes a sub-goal, ask the user. Here a dictionary plays the user, and we print every question asked.

In [ ]:
SENSORS = {"panels dusty": True, "daytime": True, "battery low": True, "dish misaligned": True,
           "night": False, "panels clean": False}

def ask(fact):
    answer = SENSORS[fact]
    print(f"  Question: is '{fact}' true? {'yes' if answer else 'no'}")
    return answer

def prove(goal, depth=0):
    pad = "  " * depth
    rules = [(n, cond) for n, (cond, concl) in enumerate(RULES, start=1) if concl == goal]
    if not rules:
        return ask(goal)
    for number, conditions in rules:
        print(f"{pad}Goal '{goal}': try rule {number}, needs {sorted(conditions)}")
        if all(prove(c, depth + 1) for c in sorted(conditions)):
            print(f"{pad}Proved '{goal}' with rule {number}")
            return True
    return False

print("Should the rover halt?", prove("emergency halt"))

Notice that backward chaining never asked about night-time: no rule on the path to 'emergency halt' depends on it.

## 4. Triples and inheritance

Knowledge graphs store facts as (subject, predicate, object). Following 'is a' links lets us inherit properties.

In [ ]:
TRIPLES = [
    ("sparrow", "is a", "bird"), ("penguin", "is a", "bird"), ("bird", "is a", "animal"),
    ("animal", "can", "breathe"), ("bird", "can", "fly"), ("penguin", "cannot", "fly"),
]

def abilities(thing):
    found, blocked = set(), {o for s, p, o in TRIPLES if s == thing and p == "cannot"}
    while thing:
        found |= {o for s, p, o in TRIPLES if s == thing and p == "can"}
        parents = [o for s, p, o in TRIPLES if s == thing and p == "is a"]
        thing = parents[0] if parents else None
    return sorted(found - blocked)

for creature in ["sparrow", "penguin"]:
    print(creature, "can:", abilities(creature))

## 5. Try it yourself

- Change `SENSORS` so it is night. Run forward chaining with the new facts. Which rules fire now?
- Add a rule of your own, such as IF 'dust storm' THEN 'panels dusty', and trace how it changes both kinds of reasoning.